# T7 STARR-FISH Analysis

This notebook analyzes the T7 STARR-FISH experimental data from mouse brain tissue. The analysis examines:
- T7 and CRE (Cis-Regulatory Element) expression patterns across cell types
- Spatial distribution of transcriptional activity
- Correlation between T7 transcripts and CRE barcodes
- Cell type-specific enhancer activity

## 1. Setup and Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="docrep")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import scvi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy.stats import linregress
from adjustText import adjust_text
import re
import sys
import os

scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

Seed set to 0


Last run with scvi-tools version: 1.3.0


In [ ]:
# Add current path to sys.path
try:
    PWD = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PWD = '/gpfs/commons/groups/ren_lab/guojiezhong/starr-fish/Mouse_brain.Guojie'
sys.path.append(f'{PWD}/')
os.chdir(PWD)

from STARRFISH import STARRFISH

## 2. Helper Functions

These utility functions support data loading, preprocessing, and analysis operations.

In [ ]:
def reload(starrfish):
    """Reload STARRFISH modules and update class references."""
    import importlib
    import STARRFISH
    import starr_fish_vae
    importlib.reload(STARRFISH)
    importlib.reload(starr_fish_vae)
    from STARRFISH import STARRFISH
    from starr_fish_vae import STARRFISHVI
    starrfish.__class__ = STARRFISH
    return starrfish

def drop_test(starrfish, test_method):
    """Remove test results from starrfish object."""
    if hasattr(starrfish, f'{test_method}_configs'):
        delattr(starrfish, f'{test_method}_configs')
    if hasattr(starrfish, f'{test_method}_results'):
        delattr(starrfish, f'{test_method}_results')
    return starrfish

In [4]:
def preprocess(adata_path):
    """Preprocess AnnData object with CRE and cell type annotations.
    
    This function:
    - Extracts FOV information from cell indices
    - Cleans subclass names
    - Processes genomic coordinates for CREs
    - Standardizes CRE naming conventions
    """
    if type(adata_path) is str:
        adata = sc.read_h5ad(adata_path)
    elif type(adata_path) is sc.AnnData:
        adata = adata_path
    
    # Extract FOV from index names
    adata.obs['fov'] = adata.obs.index.str.split('--').str[0]
    
    # Clean subclass names
    adata.obs['subclass'] = adata.obs['subclass_name'].str.replace('^[0-9]+ ', '', regex=True)
    
    # Fill empty best_subclass with label
    adata.uns['CRE_info']['best_subclass'][adata.uns['CRE_info']['best_subclass'] == ''] = \
        adata.uns['CRE_info']['label'][adata.uns['CRE_info']['best_subclass'] == ''].copy()
    
    # Process genomic coordinates
    chrom = []
    start = []
    end = []
    for i in adata.uns['CRE_info']['enh']:
        if i.startswith('chr'):
            chrom.append(i.split(':')[0])
            start.append(int(re.split('−|-', i.split(':')[1])[0]))
            end.append(int(re.split('−|-', i.split(':')[1])[1]))
        else:
            chrom.append(i)
            start.append('')
            end.append('')
    
    adata.uns['CRE_info']['Chrom'] = chrom
    adata.uns['CRE_info']['Start'] = start
    adata.uns['CRE_info']['End'] = end
    
    # Convert to string
    adata.uns['CRE_info']['Chrom'] = adata.uns['CRE_info']['Chrom'].astype(str)
    adata.uns['CRE_info']['Start'] = adata.uns['CRE_info']['Start'].astype(str)
    adata.uns['CRE_info']['End'] = adata.uns['CRE_info']['End'].astype(str)
    
    # Standardize enhancer names
    adata.uns['CRE_info']['enh'] = adata.uns['CRE_info']['Chrom'] + ':' + \
        adata.uns['CRE_info']['Start'].astype(str) + '-' + adata.uns['CRE_info']['End'].astype(str)
    
    # Format best_subclass names
    adata.uns['CRE_info']['best_subclass'] = adata.uns['CRE_info']['best_subclass'].str.replace('_', ' ')
    
    # Create standardized CRE IDs
    adata.uns['CRE_info'].index = ['CRE' + str(i+1).zfill(3) for i in range(len(adata.uns['CRE_info']))]
    adata.obsm['CRE'] = adata.obsm['CRE'][adata.uns['CRE_info'].index]
    
    if 'T7CRE' in adata.obsm.keys():
        adata.obsm['T7CRE'] = adata.obsm['T7CRE'][adata.uns['CRE_info'].index]
    
    return adata

## 3. Data Loading and Filtering

Load the STARRFISH object and apply quality control filters:
- Remove CREs with known technical issues (blacklist)
- Filter CREs with high barcode mismatch rates (>20%)
- Identify negative control CREs for normalization

In [5]:
# Load pre-processed STARRFISH object
starrfish3 = STARRFISH.load('results/starrfish3.pkl')

# Define CRE blacklist (known technical issues)
cre_blacklist = ['CRE061', 'CRE143', 'CRE001']
cre_whitelist = starrfish3.get_creinfo().index[~starrfish3.get_creinfo().index.isin(cre_blacklist)]

# Filter by barcode mismatch rate
mismatching_cres = pd.read_csv('Data/AAV_ONT_Barcode_Counts_vs_Mismatch_Percentage.csv', index_col=0)
cre_whitelist = cre_whitelist[~cre_whitelist.isin(mismatching_cres.index[mismatching_cres['MismatchPercent'] > 20])]
cre_blacklist = np.unique(cre_blacklist + mismatching_cres.index[mismatching_cres['MismatchPercent'] > 20].tolist()).tolist()

# Get negative control CREs
negative_control_cres = starrfish3.get_negative_control_cres()
negative_control_cres = [cre for cre in negative_control_cres if cre in cre_whitelist]

starrfish3.blacklist_cre = cre_blacklist

print(f"Total CREs: {len(starrfish3.get_creinfo())}")
print(f"Whitelisted CREs: {len(cre_whitelist)}")
print(f"Blacklisted CREs: {len(cre_blacklist)}")
print(f"Negative control CREs: {len(negative_control_cres)}")

Total CREs: 400
Whitelisted CREs: 389
Blacklisted CREs: 11
Negative control CREs: 7


## 4. Quality Control: CRE and T7 Counts

Examine the distribution of CRE barcodes and T7 transcripts across cells and cell types.

In [6]:
# Calculate total counts per cell
cre_counts = starrfish3.get_cre_expression().sum(axis=1)
t7_counts = starrfish3.get_t7_expression().sum(axis=1)

# Aggregate by cell type
cre_celltype = cre_counts.groupby(starrfish3.get_tag('obs:subclass')).mean()
t7_celltype = t7_counts.groupby(starrfish3.get_tag('obs:subclass')).mean()

# Plot distributions
fig, ax = plt.subplots(ncols=2, figsize=(12, 6))

# Cell type averages
sns.scatterplot(x=t7_celltype, y=cre_celltype, ax=ax[0], alpha=0.5)
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_xlabel('Average T7 counts per cell type')
ax[0].set_ylabel('Average CRE counts per cell type')

# Per-cell scatter
sns.scatterplot(x=cre_counts, y=t7_counts, ax=ax[1], alpha=0.5)
ax[1].set_xlabel('Total CRE counts per cell')
ax[1].set_ylabel('Total T7 counts per cell')

plt.tight_layout()
plt.show()

### Detection of Mismatched Cells

Identify cells with CRE barcodes but no corresponding T7 transcripts, potentially due to cell segmentation issues.

In [7]:
# Identify cells with CRE but no T7
t7_expression = starrfish3.get_t7_expression().copy()
cre_expression = starrfish3.get_cre_expression().copy()
dont_match = (t7_expression == 0) & (cre_expression > 0)

# Find the cell with the most mismatched CREs
cell_id = cre_expression[dont_match].max(axis=1).idxmax()
cre_id = cre_expression.loc[cell_id, dont_match.loc[cell_id]].idxmax()

print(f'Cell ID: {cell_id}')
print(f'CRE ID: {cre_id}')
print(f'CRE count: {cre_expression.loc[cell_id, cre_id]}')
print(f'T7 count: {t7_expression.loc[cell_id, cre_id]}')

Cell ID: Conv_zscan2_073--187
CRE ID: CRE338
CRE count: 61.0
T7 count: 0.0


### Distribution of Counts

Histogram analysis of CRE and T7 counts across all cells.

In [8]:
# Calculate unique T7 CREs per cell
t7_unique = (starrfish3.get_t7_expression() > 0).sum(axis=1)

fig, ax = plt.subplots(ncols=3, figsize=(18, 6))

# Distribution of CRE counts
sns.histplot(cre_counts[cre_counts > 0], bins=100, ax=ax[0])
ax[0].set_xlabel('CRE counts per cell')
ax[0].set_title('CRE Count Distribution')

# Distribution of T7 counts
sns.histplot(t7_counts[t7_counts > 0], bins=100, ax=ax[1])
ax[1].set_xlabel('T7 counts per cell')
ax[1].set_title('T7 Count Distribution')

# Distribution of unique T7 CREs
sns.histplot(t7_unique[t7_unique > 0], bins=50, ax=ax[2])
ax[2].set_xlabel('Unique T7 CREs per cell')
ax[2].set_title('Unique T7 CRE Distribution')

plt.tight_layout()
plt.show()

In [9]:
# Relationship between T7 and CRE counts
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=cre_counts, y=t7_counts, ax=ax, alpha=0.5)
ax.set_xlabel('CRE counts per cell')
ax.set_ylabel('T7 counts per cell')
ax.set_title('CRE vs T7 Count Relationship')
plt.show()

## 5. Cell Type Analysis

### Average T7 Counts by Cell Type

Examine the relationship between cell type abundance and average T7 expression.

In [10]:
# Calculate average T7 counts per cell type
celltype_t7 = starrfish3.get_t7_expression().sum(axis=1).groupby(starrfish3.get_celltypes()).mean()
celltype_t7 = celltype_t7.sort_values(ascending=False)
celltype_counts = starrfish3.get_celltypes().value_counts().loc[celltype_t7.index]

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=celltype_counts.values, y=celltype_t7.values, color='orange', ax=ax)
ax.set_xlabel('Number of cells')
ax.set_xscale('log')
ax.set_ylabel('Average T7 counts per cell')
ax.set_title('T7 Expression vs Cell Type Abundance')

# Annotate select cell types
cell_types = ['Oligo NN', 'Endo NN', 'Astro-NT NN', 'STR D1 Gaba', 'STR D2 Gaba', 
              'L5 IT CTX Glut', 'L6 CT CTX Glut']
texts = []
for cell_type in cell_types:
    if cell_type in celltype_t7.index:
        texts.append(ax.text(celltype_counts[cell_type], celltype_t7[cell_type], 
                            cell_type, fontsize=12, ha='right', va='bottom'))
adjust_text(texts)
plt.show()

### Load Cell Type Annotations

Import Allen Institute brain cell type nomenclature for hierarchical analysis.

In [11]:
# Calculate average counts at different taxonomic levels
subclass_T7 = starrfish3.get_t7_expression().sum(axis=1).groupby(starrfish3.get_tag('obs:subclass_name')).mean()
subclass_CRE = starrfish3.get_cre_expression().sum(axis=1).groupby(starrfish3.get_tag('obs:subclass_name')).mean()
subclass_NC_T7 = starrfish3.get_t7_expression()[negative_control_cres].sum(axis=1).groupby(
    starrfish3.get_tag('obs:subclass_name')).mean()
subclass_NC_CRE = starrfish3.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(
    starrfish3.get_tag('obs:subclass_name')).mean()

class_T7 = starrfish3.get_t7_expression().sum(axis=1).groupby(starrfish3.get_tag('obs:class_name')).mean()
class_CRE = starrfish3.get_cre_expression().sum(axis=1).groupby(starrfish3.get_tag('obs:class_name')).mean()
class_NC_T7 = starrfish3.get_t7_expression()[negative_control_cres].sum(axis=1).groupby(
    starrfish3.get_tag('obs:class_name')).mean()
class_NC_CRE = starrfish3.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(
    starrfish3.get_tag('obs:class_name')).mean()

# Load Allen Institute annotations
cluster_annotation_term = pd.read_excel('Data/abc_atlas/allen_institute_nominature.xlsx')
cluster_annotation_term = cluster_annotation_term[['class_id_label', 'subclass_id_label', 'nt_type_combo_label']]
cluster_annotation_term = cluster_annotation_term[cluster_annotation_term['subclass_id_label'].isin(subclass_T7.index)]

# Create color map for classes
unique_classes = np.sort(cluster_annotation_term['class_id_label'].unique())
palette = sns.color_palette("tab20b", n_colors=20) + sns.color_palette("tab20c", n_colors=20)[:12] + \
          sns.color_palette("tab20c", n_colors=4)[-2:]
class_color_map = dict(zip(unique_classes, palette))
cluster_annotation_term['color'] = cluster_annotation_term['class_id_label'].map(class_color_map)
cluster_annotation_term['T7Avg'] = subclass_T7.loc[cluster_annotation_term['subclass_id_label']].values

## 6. Class-Level Analysis

Visualize average T7 and CRE counts across major cell classes.

In [12]:
for p in ['T7', 'CRE', 'ratio_NC']:
    fig, ax = plt.subplots(ncols=1, figsize=(4, 6))
    
    # Prepare data with log transformation and offset
    if p == 'T7':
        original_data = class_T7.values
        data_clean = np.where(original_data > 0, original_data, 0.1)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [class_T7.mean(), starrfish3.get_t7_expression().sum(axis=1).mean()]
        ticks = [1, 2, 5, 10, 20, 40]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    elif p == 'CRE':
        original_data = class_CRE.values
        data_clean = np.where(original_data > 0, original_data, 0.05)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [class_CRE.mean(), starrfish3.get_cre_expression().sum(axis=1).mean()]
        ticks = [0.5, 1, 2, 5, 10]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    else:
        ratio = class_NC_CRE / class_NC_T7
        original_data = np.where(np.isfinite(ratio.values), ratio.values, 0)
        data_clean = np.where(original_data > 0, original_data, 0.01)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [ratio.mean(), starrfish3.get_cre_expression().sum(axis=1).mean() / 
                   starrfish3.get_t7_expression().sum(axis=1).mean()]
        ticks = [0.05, 0.1, 0.2, 0.5, 1]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    
    # Plot
    if p == 'T7':
        sns.barplot(y=class_T7.index, x=data_to_plot, ax=ax, palette=class_color_map, orient='h')
    elif p == 'CRE':
        sns.barplot(y=class_CRE.index, x=data_to_plot, ax=ax, palette=class_color_map, orient='h')
    else:
        sns.barplot(y=ratio.index, x=data_to_plot, ax=ax, palette=class_color_map, orient='h')
    
    ax.set_xlabel(f'Average {p} counts per cell')
    ax.yaxis.set_label_position('right')
    ax.yaxis.tick_right()
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Add reference lines
    for vline_x, color_x in zip(vline_xs, ['grey', 'black']):
        if vline_x > 0:
            vline_log = np.log10(vline_x) + log_offset
            ax.axvline(x=vline_log, color=color_x, linestyle='--', linewidth=1.5)
    
    ax.invert_xaxis()
    fig.tight_layout()
    fig.savefig(f'results/expr3/fig3/Avg_{p}_per_class.pdf')
    plt.show()

## 7. Neurotransmitter Type Analysis

Analyze T7 and CRE expression patterns across neurotransmitter types.

In [13]:
# Prepare neurotransmitter type labels
cluster_annotation_term['nt_type_combo_label'] = cluster_annotation_term['nt_type_combo_label'].fillna('Non-neuron')
neuron_type_label = cluster_annotation_term['nt_type_combo_label'].groupby(
    cluster_annotation_term['subclass_id_label']).first().loc[starrfish3.get_tag('obs:subclass_name')]
neuron_type_label.index = starrfish3.adata.obs.index

# Calculate averages by neurotransmitter type
neuron_type_T7 = starrfish3.get_t7_expression().sum(axis=1).groupby(neuron_type_label).mean()
neuron_type_CRE = starrfish3.get_cre_expression().sum(axis=1).groupby(neuron_type_label).mean()
neuron_type_NC_T7 = starrfish3.get_t7_expression()[negative_control_cres].sum(axis=1).groupby(neuron_type_label).mean()
neuron_type_NC_CRE = starrfish3.get_cre_expression()[negative_control_cres].sum(axis=1).groupby(neuron_type_label).mean()

# Create color map
unique_neuron_types = np.sort(cluster_annotation_term['nt_type_combo_label'].unique())
palette = sns.color_palette("terrain", n_colors=len(unique_neuron_types))
neuron_type_color_map = dict(zip(unique_neuron_types, palette))
colors = [neuron_type_color_map[x] for x in neuron_type_T7.index]

In [14]:
for p in ['T7', 'CRE', 'ratio_NC']:
    fig, ax = plt.subplots(ncols=1, figsize=(4, 6))
    
    # Prepare data
    if p == 'T7':
        original_data = neuron_type_T7.values
        data_clean = np.where(original_data > 0, original_data, 0.1)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [neuron_type_T7.mean(), starrfish3.get_t7_expression().sum(axis=1).mean()]
        ticks = [1, 2, 5, 10, 20, 40]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    elif p == 'CRE':
        original_data = neuron_type_CRE.values
        data_clean = np.where(original_data > 0, original_data, 0.05)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [neuron_type_CRE.mean(), starrfish3.get_cre_expression().sum(axis=1).mean()]
        ticks = [0.5, 1, 2, 5, 10]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    else:
        ratio = neuron_type_NC_CRE / neuron_type_NC_T7
        original_data = np.where(np.isfinite(ratio.values), ratio.values, 0)
        data_clean = np.where(original_data > 0, original_data, 0.01)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [ratio.mean(), starrfish3.get_cre_expression().sum(axis=1).mean() / 
                   starrfish3.get_t7_expression().sum(axis=1).mean()]
        ticks = [0.05, 0.1, 0.2, 0.5, 1]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    
    # Plot
    if p == 'T7':
        sns.barplot(y=neuron_type_T7.index, x=data_to_plot, ax=ax, palette=neuron_type_color_map, orient='h')
    elif p == 'CRE':
        sns.barplot(y=neuron_type_CRE.index, x=data_to_plot, ax=ax, palette=neuron_type_color_map, orient='h')
    else:
        sns.barplot(y=ratio.index, x=data_to_plot, ax=ax, palette=neuron_type_color_map, orient='h')
    
    ax.margins(y=0)
    ax.set_xlabel(f'Average {p} counts per cell')
    ax.set_ylabel('Neuron type')
    ax.yaxis.set_label_position('right')
    ax.yaxis.tick_right()
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Add reference lines
    for vline_x, color_x in zip(vline_xs, ['grey', 'black']):
        if vline_x > 0:
            vline_log = np.log10(vline_x) + log_offset
            ax.axvline(x=vline_log, color=color_x, linestyle='--', linewidth=1.5)
    
    ax.invert_xaxis()
    fig.tight_layout()
    fig.savefig(f'results/expr3/fig3/Avg_{p}_per_neuron_type.pdf')
    plt.show()

## 8. Subclass-Level Analysis

High-resolution analysis at the cell subclass level.

In [15]:
# Create color map for subclasses
subclass_color_map = dict(zip(cluster_annotation_term['subclass_id_label'], cluster_annotation_term['color']))
colors = [subclass_color_map[x] for x in subclass_T7.index]

for p in ['T7', 'CRE', 'ratio_NC']:
    fig, ax = plt.subplots(ncols=1, figsize=(4, 15))
    
    # Prepare data
    if p == 'T7':
        original_data = subclass_T7.values
        data_clean = np.where(original_data > 0, original_data, 0.1)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [subclass_T7.mean(), starrfish3.get_t7_expression().sum(axis=1).mean()]
        ticks = [1, 2, 5, 10, 20, 40]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    elif p == 'CRE':
        original_data = subclass_CRE.values
        data_clean = np.where(original_data > 0, original_data, 0.05)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [subclass_CRE.mean(), starrfish3.get_cre_expression().sum(axis=1).mean()]
        ticks = [0.5, 1, 2, 5, 10]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    else:
        ratio = subclass_NC_CRE / subclass_NC_T7
        original_data = np.where(np.isfinite(ratio.values), ratio.values, 0)
        data_clean = np.where(original_data > 0, original_data, 0.01)
        data_log = np.log10(data_clean)
        log_offset = -np.min(data_log) + 0.1
        data_to_plot = data_log + log_offset
        vline_xs = [subclass_NC_CRE.mean() / subclass_NC_T7.mean(), 
                   starrfish3.get_cre_expression().sum(axis=1).mean() / 
                   starrfish3.get_t7_expression().sum(axis=1).mean()]
        ticks = [0.05, 0.1, 0.2, 0.5, 1]
        tick_labels = ticks
        tick_positions = np.log10(ticks) + log_offset
    
    # Plot
    if p == 'T7':
        ax.barh(y=subclass_T7.index, width=data_to_plot, height=1.0, color=colors)
    elif p == 'CRE':
        ax.barh(y=subclass_CRE.index, width=data_to_plot, height=1.0, color=colors)
    else:
        ax.barh(y=ratio.index, width=data_to_plot, height=1.0, color=colors)
    
    ax.margins(y=0)
    ax.set_xlabel(f'Average {p} counts per cell')
    ax.set_yticklabels([])
    ax.set_yticks([])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Set x-axis limits
    x_min = np.min(data_to_plot) - 0.1
    x_max = np.max(data_to_plot) + 0.1
    ax.set_xlim(x_min, x_max)
    
    # Add reference lines
    for vline_x, color_x in zip(vline_xs, ['grey', 'black']):
        if vline_x > 0:
            vline_log = np.log10(vline_x) + log_offset
            ax.axvline(x=vline_log, color=color_x, linestyle='--', linewidth=1.5)
    
    ax.invert_xaxis()
    ax.set_ylim(ax.get_ylim()[::-1])
    
    fig.tight_layout()
    fig.savefig(f'results/expr3/fig3/Avg_{p}_per_subclass.pdf')
    plt.show()

## 9. Spatial Visualization

### T7 Expression Spatial Patterns

Visualize the spatial distribution of average T7 expression across the tissue.

In [16]:
# Subclass-level T7 spatial distribution
starrfish3.adata.obsm['T7Avg'] = pd.DataFrame(
    {'T7Avg': subclass_T7.loc[starrfish3.adata.obs['subclass_name']].values},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('T7Avg', use='T7Avg', 
                           norm_by_negative_control_cell_type_sum=False, 
                           norm_by_t7_cell_type_mean=False,
                           log=False, sz_min=1, sz_max=1)
fig.savefig('results/expr3/fig3/Spatial_subclass_T7Avg.pdf')
plt.show()

In [17]:
# Class-level T7 spatial distribution
starrfish3.adata.obsm['T7Avg'] = pd.DataFrame(
    {'T7Avg': class_T7.loc[starrfish3.adata.obs['class_name']].values},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('T7Avg', use='T7Avg', 
                           norm_by_negative_control_cell_type_sum=False, 
                           norm_by_t7_cell_type_mean=False,
                           log=False, sz_min=1, sz_max=1)
fig.savefig('results/expr3/fig3/Spatial_class_T7Avg.pdf')
plt.show()

In [18]:
# Neurotransmitter-level T7 spatial distribution
starrfish3.adata.obsm['T7Avg'] = pd.DataFrame(
    {'T7Avg': neuron_type_T7.loc[neuron_type_label.values].values},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('T7Avg', use='T7Avg', 
                           norm_by_negative_control_cell_type_sum=False, 
                           norm_by_t7_cell_type_mean=False,
                           log=False, sz_min=1, sz_max=1)
fig.savefig('results/expr3/fig3/Spatial_neurontransmitter_T7Avg.pdf')
plt.show()

In [19]:
# Total T7 sum per cell with class colors
starrfish3.adata.obsm['T7Sum'] = pd.DataFrame(
    {'T7Sum': starrfish3.get_t7_expression().sum(axis=1)},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('T7Sum', use='T7Sum', 
                           norm_by_negative_control_cell_type_sum=False, 
                           norm_by_t7_cell_type_mean=False,
                           cell_types_tag='obs:class_name',
                           cell_types_to_visualize=starrfish3.adata.obs['class_name'].unique().tolist(),
                           use_celltype_cmap=True,
                           celltype_cmap=class_color_map,
                           log=False, sz_min=1, sz_max=1)
fig.savefig('results/expr3/fig3/Spatial_class_T7Sum.pdf')
plt.show()

### CRE Expression Spatial Patterns

In [20]:
# Subclass-level CRE spatial distribution
starrfish3.adata.obsm['CREAvg'] = pd.DataFrame(
    {'CREAvg': subclass_CRE.loc[starrfish3.adata.obs['subclass_name']].values},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('CREAvg', use='CREAvg', 
                           norm_by_negative_control_cell_type_sum=False, log=False,
                           norm_by_t7_cell_type_mean=False,
                           sz_min=1, sz_max=1)
fig.savefig('results/expr3/fig3/Spatial_subclass_CREAvg.pdf')
plt.show()

In [21]:
# Class-level CRE spatial distribution
starrfish3.adata.obsm['CREAvg'] = pd.DataFrame(
    {'CREAvg': class_CRE.loc[starrfish3.adata.obs['class_name']].values},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('CREAvg', use='CREAvg', 
                           norm_by_negative_control_cell_type_sum=False, log=False,
                           norm_by_t7_cell_type_mean=False,
                           sz_min=1, sz_max=1)
fig.savefig('results/expr3/fig3/Spatial_class_CREAvg.pdf')
plt.show()

In [22]:
# Neurotransmitter-level CRE spatial distribution
starrfish3.adata.obsm['CREAvg'] = pd.DataFrame(
    {'CREAvg': neuron_type_CRE.loc[neuron_type_label.values].values},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('CREAvg', use='CREAvg', 
                           norm_by_negative_control_cell_type_sum=False, log=False,
                           norm_by_t7_cell_type_mean=False,
                           sz_min=1, sz_max=1)
fig.savefig('results/expr3/fig3/Spatial_neurontransmitter_CREAvg.pdf')
plt.show()

### CRE/T7 Ratio Spatial Patterns

Visualize the ratio of CRE barcodes to T7 transcripts using negative control CREs.

In [23]:
# Class-level CRE/T7 ratio
starrfish3.adata.obsm['CRE/T7'] = pd.DataFrame(
    {'CRE/T7': (class_NC_CRE / class_NC_T7).loc[starrfish3.adata.obs['class_name']].values},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('CRE/T7', use='CRE/T7', 
                           norm_by_negative_control_cell_type_sum=False, 
                           norm_by_t7_cell_type_mean=False,
                           log=False, sz_min=1, sz_max=1, nmax=2)
fig.savefig('results/expr3/fig3/Spatial_CRET7_class_NC_ratio.pdf')
plt.show()

In [24]:
# Subclass-level CRE/T7 ratio
starrfish3.adata.obsm['CRE/T7'] = pd.DataFrame(
    {'CRE/T7': (subclass_NC_CRE / subclass_NC_T7).loc[starrfish3.adata.obs['subclass_name']].values},
    index=starrfish3.adata.obs.index)
fig = starrfish3.plot_gene('CRE/T7', use='CRE/T7', 
                           norm_by_negative_control_cell_type_sum=False, 
                           norm_by_t7_cell_type_mean=False,
                           log=False, sz_min=1, sz_max=1, nmax=2)
fig.savefig('results/expr3/fig3/Spatial_CRET7_subclass_NC_ratio.pdf')
plt.show()

## 10. Technical Validation

### Cell Detection Rates

In [25]:
# Calculate proportion of cells with CRE and T7 detection
total_cells = starrfish3.adata.shape[0]
cre_positive = (cre_counts > 0).sum()
t7_positive = (t7_counts > 0).sum()

print(f"Proportion of cells with CRE: {cre_positive / total_cells:.3f}")
print(f"Proportion of cells with T7: {t7_positive / total_cells:.3f}")
print(f"T7/CRE detection ratio: {t7_positive / cre_positive:.3f}")

Proportion of cells with CRE: 0.288
Proportion of cells with T7: 0.614
T7/CRE detection ratio: 2.136


## 11. Correlation Analysis

### T7 Correlation with AAV Library Size

Examine whether T7 detection is biased by AAV library representation.

In [26]:
# Calculate total T7 counts per CRE
t7_counts = starrfish3.get_t7_expression().sum(axis=0)
t7_counts = t7_counts.loc[starrfish3.lib_size.index]

fig, ax = plt.subplots(ncols=2, figsize=(12, 6))

# Raw scale
sns.scatterplot(x=(starrfish3.lib_size['counts']), y=t7_counts, ax=ax[0], alpha=0.5)
ax[0].set_xlabel('AAV library size')
ax[0].set_ylabel('Total T7 counts in all cells')

slope, intercept, r_value, p_value, std_err = linregress(starrfish3.lib_size['counts'], t7_counts)
x = np.linspace(starrfish3.lib_size['counts'].min(), starrfish3.lib_size['counts'].max(), 100)
y = slope * x + intercept
ax[0].plot(x, y, color='red', label=f'Correlation: {r_value:.2f}, p-value: {p_value:.2e}')
ax[0].legend()

# Log scale
t7_counts_log = np.log1p(t7_counts)
sns.scatterplot(x=np.log1p(starrfish3.lib_size['counts']), y=t7_counts_log, ax=ax[1], alpha=0.5)
ax[1].set_xlabel('Log(AAV library size)')
ax[1].set_ylabel('Log(T7 counts)')

slope_log, intercept_log, r_value_log, p_value_log, std_err_log = linregress(
    np.log1p(starrfish3.lib_size['counts']), t7_counts_log)
x_log = np.linspace(np.log1p(starrfish3.lib_size['counts']).min(), 
                    np.log1p(starrfish3.lib_size['counts']).max(), 100)
y_log = slope_log * x_log + intercept_log
ax[1].plot(x_log, y_log, color='red', label=f'Correlation: {r_value_log:.2f}, p-value: {p_value_log:.2e}')
ax[1].legend()

fig.savefig('results/expr3/fig3/T7_vs_AAV_library_size.pdf')
plt.show()

### Cell Type-Specific Library Size Correlation

In [27]:
# Calculate correlation per cell type
corr_df = pd.DataFrame(columns=['celltype', 'correlation', 'p_value'])
for celltype in starrfish3.get_celltypes().unique():
    t7_counts = starrfish3.get_t7_expression().loc[starrfish3.get_celltypes() == celltype].sum(axis=0)
    t7_counts = t7_counts.loc[starrfish3.lib_size.index]
    slope, intercept, r_value, p_value, std_err = linregress(
        starrfish3.lib_size['counts'], np.log1p(t7_counts))
    corr_df = pd.concat([corr_df, pd.DataFrame({
        'celltype': [celltype], 'correlation': [r_value], 'p_value': [p_value]
    })], ignore_index=True)

corr_df = corr_df.sort_values(by='correlation', ascending=False)
cell_counts = starrfish3.get_celltypes().value_counts()
corr_df['cell_counts'] = corr_df['celltype'].map(cell_counts)

# Plot
fig, ax = plt.subplots(ncols=2, figsize=(12, 6))
sns.histplot(data=corr_df, x='correlation', ax=ax[0], bins=30, kde=True)
ax[0].set_xlabel('Correlation with AAV library size')

sns.scatterplot(data=corr_df, x='cell_counts', y='correlation', ax=ax[1], alpha=0.5)
ax[1].set_xlabel('Number of cells')
ax[1].set_xscale('log')
ax[1].set_ylabel('Correlation with AAV library size')

fig.savefig('results/expr3/fig3/T7_AAV_library_size_correlation_per_celltype.pdf')
plt.show()

### T7 vs CRE Count Correlation

Test whether T7 transcript counts correlate with CRE barcode counts across all CREs.

In [28]:
# Overall correlation
fig, ax = plt.subplots(figsize=(5, 4))
cre_counts = starrfish3.get_cre_expression().sum(axis=0)
t7_counts = starrfish3.get_t7_expression().sum(axis=0)
sns.scatterplot(x=np.log(cre_counts), y=np.log(t7_counts), ax=ax, alpha=0.5)

slope, intercept, r_value, p_value, std_err = linregress(np.log(cre_counts), np.log(t7_counts))
x = np.linspace(np.log(cre_counts).min(), np.log(cre_counts).max(), 100)
y = slope * x + intercept
ax.plot(x, y, color='red', label=f'Correlation: {r_value:.2f}, p-value: {p_value:.2e}')
ax.legend()
ax.set_xlabel('Total CRE counts per CRE (log scale)')
ax.set_ylabel('Total T7 counts per CRE (log scale)')
fig.savefig('results/expr3/fig3/Total_T7_vs_Total_CRE_counts_per_CRE.pdf')
plt.show()

### Cell Type-Specific T7/CRE Correlation

In [29]:
# Calculate correlation per cell type
corr_df = pd.DataFrame(columns=['celltype', 'correlation', 'p_value'])
for celltype in starrfish3.get_celltypes().unique():
    cre_counts = starrfish3.get_cre_expression().loc[starrfish3.get_celltypes() == celltype].sum(axis=0)
    t7_counts = starrfish3.get_t7_expression().loc[starrfish3.get_celltypes() == celltype].sum(axis=0)
    try:
        slope, intercept, r_value, p_value, std_err = linregress(
            np.log(cre_counts + 1), np.log(t7_counts + 1))
        corr_df = pd.concat([corr_df, pd.DataFrame({
            'celltype': [celltype], 'correlation': [r_value], 'p_value': [p_value]
        })], ignore_index=True)
    except Exception as e:
        print(f"Error processing celltype {celltype}: {e}")

corr_df = corr_df.sort_values(by='correlation', ascending=False)
cell_counts = starrfish3.get_celltypes().value_counts()
corr_df['cell_counts'] = corr_df['celltype'].map(cell_counts)

# Plot
fig, ax = plt.subplots(ncols=2, figsize=(12, 6))
sns.histplot(data=corr_df, x='correlation', ax=ax[0], bins=30, kde=True)
ax[0].set_xlabel('Correlation between total T7 and total CRE counts')

sns.scatterplot(data=corr_df, x='cell_counts', y='correlation', ax=ax[1], alpha=0.5)
ax[1].set_xlabel('Number of cells')
ax[1].set_xscale('log')
ax[1].set_ylabel('Correlation between total T7 and total CRE counts')

fig.savefig('results/expr3/fig3/T7_CRE_total_counts_correlation_per_celltype.pdf')
plt.show()

Error processing celltype SPVC Nmu Glut: Cannot calculate a linear regression if all x values are identical
Error processing celltype COAp Grxcr2 Glut: Cannot calculate a linear regression if all x values are identical
Error processing celltype PVHd Gsc Gaba: Cannot calculate a linear regression if all x values are identical
Error processing celltype IO Fgl2 Glut: Cannot calculate a linear regression if all x values are identical
Error processing celltype MV Xdh Gly-Gaba: Cannot calculate a linear regression if all x values are identical
Error processing celltype MG-POL-SGN Nts Glut: Cannot calculate a linear regression if all x values are identical
Error processing celltype CLA-EPd-CTX Car3 Glut: Cannot calculate a linear regression if all x values are identical
Error processing celltype NLOT Rho Glut: Cannot calculate a linear regression if all x values are identical
